# Clarity With Data (CWD) | Titanic: Machine Learning with Meaning

In [ ]:
# Clarity With Data | Titanic ML From Disaster
# Mission: Predict survival AND tell the data's human story

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import shap

In [ ]:
train = pd.read_csv('/kaggle/input/titanic/train.csv')
test = pd.read_csv('/kaggle/input/titanic/test.csv')

In [ ]:
#Exploratory Data Analysis
train.columns

In [ ]:
sns.barplot(x='Sex', y='Survived', data=train)
plt.title("Survival Rate by Gender")
plt.show()

train['AgeGroup'] = pd.cut(train['Age'], bins=[0,10,20,30,40,50,60,70,80], right=False)
sns.barplot(x='AgeGroup', y='Survived', data=train, errorbar=None)
plt.title("Average Survival Rate by Age Group")
plt.xticks(rotation=45)
plt.xlabel("Age Group")
plt.ylabel("Survival Rate")
plt.show()

sns.barplot(x='Pclass', y='Survived', data=train)
plt.title("Survival Rate by Class")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# -------- Feature engineering done BEFORE the pipeline (works for both train/test) --------
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    # collapse rare titles to reduce sparsity
    rare = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
    df['Title'] = df['Title'].replace(
        {'Mlle':'Miss', 'Ms':'Miss', 'Mme':'Mrs', **{t:'Rare' for t in rare}}
    )
    return df

train_fe = add_features(train)

# -------- Select features --------
num_feats = ['Age','Fare','Pclass','FamilySize','IsAlone','SibSp','Parch']
cat_feats = ['Sex','Embarked','Title']  # One-hot these

X = train_fe[num_feats + cat_feats]
y = train_fe['Survived']

# -------- Preprocessors --------
numeric_tf = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_tf = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_tf, num_feats),
        ('cat', categorical_tf, cat_feats)
    ]
)

# -------- Model pipeline --------
clf = Pipeline(steps=[
    ('prep', preprocess),
    ('model', RandomForestClassifier(
        n_estimators=300, max_depth=None, random_state=42, n_jobs=-1
    ))
])

# -------- Train/validate --------
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf.fit(X_train, y_train)
pred = clf.predict(X_valid)

print("Accuracy:", accuracy_score(y_valid, pred))
print(classification_report(y_valid, pred))

# CWD Survival Scorecard (0-100)+Categories

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_recall_curve, roc_curve, auc,
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, brier_score_loss
)
import matplotlib.pyplot as plt

# --- Use existing validation probabilities ---
USE_CALIBRATED = False #<- set to true if want to use cal_proba instead

proba = cal_proba if USE_CALIBRATED else valid_proba  # from earlier: clf.predict_proba(X_valid)[:,1]
y_true = y_valid.values

# --- Evaluate metrics across thresholds ---
thresholds = np.linspace(0.00, 1.00, 201)
rows = []
for t in thresholds:
    y_hat = (proba >= t).astype(int)
    rows.append({
        "threshold": t,
        "precision": precision_score(y_true, y_hat, zero_division=0),
        "recall": recall_score(y_true, y_hat, zero_division=0),
        "f1": f1_score(y_true, y_hat, zero_division=0)
    })
th_df = pd.DataFrame(rows)

# --- Best F1 threshold ---
best_f1_idx = th_df['f1'].idxmax()
best_f1_thr = float(th_df.loc[best_f1_idx, 'threshold'])
best_f1_row = th_df.loc[best_f1_idx]

print(f"Best F1 threshold = {best_f1_thr:.3f} | "
      f"P={best_f1_row.precision:.3f} R={best_f1_row.recall:.3f} F1={best_f1_row.f1:.3f}")

# --- Option: choose threshold to hit a recall target (e.g., >= 0.80) ---
RECALL_TARGET = 0.80
cand = th_df[th_df['recall'] >= RECALL_TARGET]
if not cand.empty:
    # among those, choose the one with best precision (or best F1)
    best_recall_thr = float(cand.sort_values(['precision','f1'], ascending=False).iloc[0]['threshold'])
    print(f"Threshold for ≥{RECALL_TARGET:.0%} recall = {best_recall_thr:.3f}")
else:
    best_recall_thr = None
    print(f"No threshold reaches recall ≥ {RECALL_TARGET:.0%} on this split.")

# --- Plots: PR curve, ROC, and metric-vs-threshold ---
prec, rec, pr_th = precision_recall_curve(y_true, proba)
fpr, tpr, _ = roc_curve(y_true, proba)

plt.figure()
plt.plot(rec, prec)
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

plt.figure()
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title(f"ROC Curve (AUC={auc(fpr,tpr):.3f})")
plt.show()

plt.figure()
plt.plot(thresholds, th_df['f1'], label='F1')
plt.plot(thresholds, th_df['precision'], label='Precision')
plt.plot(thresholds, th_df['recall'], label='Recall')
plt.axvline(best_f1_thr, linestyle='--', label='Best F1')
if best_recall_thr is not None:
    plt.axvline(best_recall_thr, linestyle=':', label=f'Recall≥{RECALL_TARGET:.0%}')
plt.xlabel("Threshold"); plt.ylabel("Score")
plt.title("Metrics vs Threshold"); plt.legend()
plt.show()

# --- Lock in operating point (choose one) ---
# --- Lock in operating point (choose one) ---
THRESH = float(best_f1_thr)          # or: best_recall_thr
OPERATING = {"use_calibrated": USE_CALIBRATED, "threshold": THRESH}

print(f"\n[OPERATING] calibrated={OPERATING['use_calibrated']}  threshold={OPERATING['threshold']:.3f}")

y_pred_thresh = (proba >= THRESH).astype(int)
print(f"\nUsing threshold = {THRESH:.3f}")
print(classification_report(y_true, y_pred_thresh))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred_thresh))

In [ ]:
import numpy as np
import pandas as pd

# scores are already in valid_score (0–100); make bands robustly
bands = ['Very Low','Low','Uncertain','Moderate','High']

def make_bands(scores):
    s = np.asarray(scores, dtype=float)
    s = np.clip(s, 0, 100)
    # widen edges so nothing sits exactly on a boundary
    return pd.cut(s, bins=[-0.5, 20, 40, 60, 80, 100.5],
                  labels=bands, include_lowest=True)

# --- Probabilities and scoring ---
valid_proba = clf.predict_proba(X_valid)[:, 1]
valid_score = (valid_proba * 100).round(1)
valid_band  = make_bands(valid_score)

# --- Build tidy DataFrame ---
score_df = X_valid.copy()
score_df = score_df.assign(
    Survived=y_valid.values,
    CWD_Score=valid_score,
    CWD_Band=valid_band
)

# --- Attach predicted label using tuned threshold ---
score_df = score_df.assign(
    CWD_Label=(valid_proba >= OPERATING["threshold"]).astype(int)
)

# Quick peek
score_df.head()

In [ ]:
score_df = score_df.assign(
    Survived=y_valid.values,
    CWD_Score=valid_score,
    CWD_Band=valid_band
)

# Quick peek
score_df.head()

In [ ]:
# Distribution of scores by true outcome
sns.histplot(data=score_df, x='CWD_Score', hue='Survived', bins=20, multiple='stack')
plt.title('CWD Survival Score Distribution (Valid Set)')
plt.xlabel('CWD Survival Score (0–100)')
plt.ylabel('Count')
plt.show()

# Average score by band and gender/class
fig = plt.figure()
sns.barplot(data=score_df, x='CWD_Band', y='CWD_Score', estimator=np.mean, errorbar=None)
plt.title('Average Score by CWD Band')
plt.ylabel('Average Score')
plt.show()

fig = plt.figure()
sns.barplot(data=score_df, x='Sex', y='CWD_Score', errorbar=None)
plt.title('Average Score by Sex')
plt.show()

fig = plt.figure()
sns.barplot(data=score_df, x='Pclass', y='CWD_Score', errorbar=None)
plt.title('Average Score by Passenger Class')
plt.show()

In [ ]:
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

# --- Calibration curve for current model ---
prob_true, prob_pred = calibration_curve(y_true, proba, n_bins=10, strategy='quantile')

plt.figure()
plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0,1],[0,1],'--')
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Calibration Curve (uncalibrated)")
plt.show()

print("Brier score (lower is better):", brier_score_loss(y_true, proba))

# --- OPTIONAL: fit an isotonic-calibrated version of the full pipeline ---
# This will refit internally using CV; safe to keep your original 'clf' untouched.
cal_clf = CalibratedClassifierCV(base_estimator=clf, method='isotonic', cv=5)
cal_clf.fit(X_train, y_train)

cal_proba = cal_clf.predict_proba(X_valid)[:,1]
prob_true_c, prob_pred_c = calibration_curve(y_true, cal_proba, n_bins=10, strategy='quantile')

plt.figure()
plt.plot(prob_pred, prob_true, marker='o', label='Uncalibrated')
plt.plot(prob_pred_c, prob_true_c, marker='o', label='Isotonic Calibrated')
plt.plot([0,1],[0,1],'--', label='Perfect')
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Calibration Curve Comparison")
plt.legend()
plt.show()

print("Brier score (uncalibrated):", brier_score_loss(y_true, proba))
print("Brier score (calibrated):  ", brier_score_loss(y_true, cal_proba))

In [ ]:
# Ensure numerics
score_df['Age_num']  = pd.to_numeric(score_df['Age'],  errors='coerce')
score_df['Fare_num'] = pd.to_numeric(score_df['Fare'], errors='coerce')

# For visuals that require finite values, drop rows where Age or Fare is NaN
viz_df = score_df.dropna(subset=['Age_num','Fare_num']).copy()

In [ ]:
# Age distribution by survival (no warnings)
sns.histplot(data=viz_df, x='Age_num', hue='Survived', bins=20, multiple='stack')
plt.title("Age Distribution by Survival"); plt.xlabel("Age"); plt.ylabel("Count"); plt.show()

# Heatmap: Scores by Age × Fare (no warnings)
viz_df['AgeBin']  = pd.cut(viz_df['Age_num'],  bins=[0,10,20,30,40,50,60,70,80], right=False)
viz_df['FareBin'] = pd.qcut(viz_df['Fare_num'], q=6, duplicates='drop')
pivot = viz_df.pivot_table(index='AgeBin', columns='FareBin', values='CWD_Score', aggfunc='mean')
sns.heatmap(pivot)
plt.title('Mean CWD Score by Age × Fare'); plt.xlabel('Fare (quantiles)'); plt.ylabel('Age (bins)'); plt.show()

In [ ]:
# ===================== KAGGLE SUBMISSION (final) =====================
import os
import numpy as np
import pandas as pd

# ---- 1) Build test features (same transforms as train) ----
test_fe = add_features(test)                 # uses your helper from earlier
test_X  = test_fe[X.columns]                 # align exactly to training columns

# ---- 2) Choose probability source (calibrated or not) ----
use_calibrated = False
if 'OPERATING' in globals():
    use_calibrated = bool(OPERATING.get('use_calibrated', False))

if use_calibrated and 'cal_clf' in globals():
    test_proba = cal_clf.predict_proba(test_X)[:, 1]
else:
    test_proba = clf.predict_proba(test_X)[:, 1]

# ---- 3) Apply tuned threshold to get labels ----
threshold = 0.50
if 'OPERATING' in globals():
    threshold = float(OPERATING.get('threshold', 0.50))

test_pred = (test_proba >= threshold).astype(int)

# ---- 4) Write submission to /kaggle/working ----
SUBMISSION_PATH = "/kaggle/working/submission.csv"
submission = pd.DataFrame({
    "PassengerId": test_fe["PassengerId"].astype(int),
    "Survived":    test_pred.astype(int)
})
submission.to_csv(SUBMISSION_PATH, index=False)

# ---- 5) Sanity checks & preview ----
print("Wrote:", SUBMISSION_PATH)
print("Exists? ", os.path.exists(SUBMISSION_PATH))
print("Size   : ", os.path.getsize(SUBMISSION_PATH), "bytes")
print("Head:\n", pd.read_csv(SUBMISSION_PATH).head(), "\n")
print(f"calibrated={use_calibrated} | threshold={threshold:.3f}")